In [1]:
!pip install -q transformers==4.46.3 datasets scikit-learn kagglehub scipy

import json, random
from pathlib import Path
from scipy.stats import spearmanr
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import kagglehub
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import roc_auc_score
from transformers import CLIPModel, CLIPProcessor

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 82.0 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 68.3 MB/s eta 0:00:00:00:01


2026-06-07 19:50:00.754373: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780861800.989416      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780861801.062075      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780861801.621897      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780861801.621934      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780861801.621937      58 computation_placer.cc:177] computation placer alr

Device: cuda


In [3]:
coco_path = Path(kagglehub.dataset_download("awsaf49/coco-2017-dataset"))
COCO_IMG  = coco_path / "coco2017/val2017"
COCO_ANN  = coco_path / "coco2017/annotations/captions_val2017.json"

with open(COCO_ANN) as f: coco = json.load(f)
id2path = {img["id"]: COCO_IMG/img["file_name"] for img in coco["images"]}
random.shuffle(coco["annotations"])
seen, pairs = set(), []
for ann in coco["annotations"]:
    if ann["image_id"] not in seen and id2path[ann["image_id"]].exists():
        pairs.append((ann["image_id"], ann["caption"])); seen.add(ann["image_id"])
    if len(pairs) >= 5000: break

images = [Image.open(id2path[i]).convert("RGB") for i, _ in tqdm(pairs)]
texts  = [c for _, c in pairs]
print(f"{len(pairs)} COCO pairs loaded")

@torch.no_grad()
def encode(imgs, txts, model, proc, bs=64):
    zi, zt = [], []
    for i in range(0, len(imgs), bs):
        inp = proc(text=txts[i:i+bs], images=imgs[i:i+bs],
                   return_tensors="pt", padding=True, truncation=True, max_length=77).to(DEVICE)
        out = model(**inp)
        zi.append(out.image_embeds.cpu()); zt.append(out.text_embeds.cpu())
    return torch.cat(zi), torch.cat(zt)

print("Encoding ViT-B/32..."); Zi_b, Zt_b = encode(images, texts, model_b32, proc_b32)
print("Encoding ViT-L/14..."); Zi_l, Zt_l = encode(images, texts, model_l14, proc_l14)
print(f"Done. Zi_b={tuple(Zi_b.shape)}  Zi_l={tuple(Zi_l.shape)}")

100%|██████████| 5000/5000 [00:47<00:00, 104.58it/s]


5000 COCO pairs loaded
Encoding ViT-B/32...
Encoding ViT-L/14...
Done. Zi_b=(5000, 512)  Zi_l=(5000, 768)


In [4]:
def D_score(zi, zt):
    vi = F.normalize(zi, dim=-1); vt = F.normalize(zt, dim=-1)
    return (vi - vt).abs().sum(-1) / (vi.shape[-1] ** 0.5)

D_b = D_score(Zi_b, Zt_b).numpy()
D_l = D_score(Zi_l, Zt_l).numpy()
N_PAIRS = len(D_b)

HARD_FRAC = 0.30
k_hard    = int(N_PAIRS * HARD_FRAC)
hard_pool = np.argpartition(D_b, -k_hard)[-k_hard:].tolist()

N_TRAIN = 1000
random.seed(42)
hard_idx  = random.sample(hard_pool, N_TRAIN)
rand_idx  = random.sample(range(N_PAIRS), N_TRAIN)
train_used = set(hard_idx) | set(rand_idx)

eval_pool = [i for i in range(N_PAIRS) if i not in train_used]
random.seed(42); random.shuffle(eval_pool)

print(f"Hard pool:   {len(hard_pool)} pairs (top {HARD_FRAC*100:.0f}%)")
print(f"Train hard:  {len(hard_idx)}")
print(f"Train rand:  {len(rand_idx)}")
print(f"Eval pool:   {len(eval_pool)} held-out pairs")

Hard pool:   1500 pairs (top 30%)
Train hard:  1000
Train rand:  1000
Eval pool:   3192 held-out pairs


In [6]:
class JEPA(nn.Module):
    def __init__(self, dim=512, hidden=512):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, hidden), nn.GELU(), nn.LayerNorm(hidden),
            nn.Linear(hidden, hidden), nn.GELU(), nn.LayerNorm(hidden),
            nn.Linear(hidden, dim)
        )
    def forward(self, z): return self.net(z)
    @torch.no_grad()
    def error(self, zv, zt):
        return 1 - F.cosine_similarity(self(zv), zt, dim=-1)

def train_jepa(zi_tr, zt_tr, dim=512, epochs=200, lr=5e-4, seed=42):
    torch.manual_seed(seed); np.random.seed(seed)
    m   = JEPA(dim).to(DEVICE)
    opt = torch.optim.Adam(m.parameters(), lr=lr, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    zv  = zi_tr.to(DEVICE); zt = zt_tr.to(DEVICE)
    for ep in range(1, epochs + 1):
        p    = torch.randperm(len(zv))
        loss = (1 - F.cosine_similarity(m(zv[p]), zt[p].detach(), dim=-1)).mean()
        opt.zero_grad(); loss.backward(); opt.step(); sch.step()
    return m.eval().cpu()

# Best seed from NB1 was 13
BEST_SEED = 13
torch.manual_seed(BEST_SEED); np.random.seed(BEST_SEED); random.seed(BEST_SEED)
h_idx = random.sample(hard_pool, N_TRAIN)
r_idx = random.sample(range(N_PAIRS), N_TRAIN)

print(f"Training pred_hard (seed={BEST_SEED}, N={N_TRAIN}, epochs=200)...")
pred_hard = train_jepa(Zi_b[h_idx], Zt_b[h_idx], epochs=200, seed=BEST_SEED)
print(f"Training pred_rand  (seed={BEST_SEED}, N={N_TRAIN}, epochs=200)...")
pred_rand = train_jepa(Zi_b[r_idx], Zt_b[r_idx], epochs=200, seed=BEST_SEED)

torch.save(pred_hard.state_dict(), "/kaggle/working/pred_hard.pt")
torch.save(pred_rand.state_dict(),  "/kaggle/working/pred_rand.pt")
print("Predictors trained and saved.")

Training pred_hard (seed=13, N=1000, epochs=200)...
Training pred_rand  (seed=13, N=1000, epochs=200)...
Predictors trained and saved.


In [7]:
# ── Canonical Projection Head: Linear 768→512 + LayerNorm ────────────────
class ProjHead(nn.Module):
    def __init__(self, in_dim=768, out_dim=512):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, out_dim), nn.LayerNorm(out_dim))
    def forward(self, z): return self.net(z)

def train_proj(epochs=300, lr=1e-3, seed=42):
    torch.manual_seed(seed)
    proj = ProjHead().to(DEVICE)
    opt  = torch.optim.Adam(proj.parameters(), lr=lr)
    sch  = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    zin  = Zi_l.to(DEVICE); zio = Zi_b.to(DEVICE)
    ztn  = Zt_l.to(DEVICE); zto = Zt_b.to(DEVICE)
    for ep in range(1, epochs + 1):
        li   = (1 - F.cosine_similarity(proj(zin), zio, dim=-1)).mean()
        lt   = (1 - F.cosine_similarity(proj(ztn), zto, dim=-1)).mean()
        loss = (li + lt) / 2
        opt.zero_grad(); loss.backward(); opt.step(); sch.step()
        if ep % 100 == 0: print(f"  ep {ep}  loss={loss.item():.4f}")
    return proj.eval().cpu()

print("Training canonical projection head (L/14 768d → B/32 512d, 300 epochs)...")
proj_head = train_proj()
torch.save(proj_head.state_dict(), "/kaggle/working/proj_head.pt")
print("Saved proj_head.pt\n")

# ── L/14 Direct Retrain: JEPA(768) trained on L/14 embeddings ────────────
# This is the missing baseline from the paper:
# does canonical projection add value vs just retraining on L/14 directly?
class JEPA768(nn.Module):
    def __init__(self, hidden=512):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(768, hidden), nn.GELU(), nn.LayerNorm(hidden),
            nn.Linear(hidden, hidden), nn.GELU(), nn.LayerNorm(hidden),
            nn.Linear(hidden, 768)
        )
    def forward(self, z): return self.net(z)
    @torch.no_grad()
    def error(self, zv, zt):
        return 1 - F.cosine_similarity(self(zv), zt, dim=-1)

def train_jepa768(zi_tr, zt_tr, epochs=200, lr=5e-4, seed=42):
    torch.manual_seed(seed); np.random.seed(seed)
    m   = JEPA768().to(DEVICE)
    opt = torch.optim.Adam(m.parameters(), lr=lr, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    zv  = zi_tr.to(DEVICE); zt = zt_tr.to(DEVICE)
    for ep in range(1, epochs + 1):
        p    = torch.randperm(len(zv))
        loss = (1 - F.cosine_similarity(m(zv[p]), zt[p].detach(), dim=-1)).mean()
        opt.zero_grad(); loss.backward(); opt.step(); sch.step()
    return m.eval().cpu()

print("Training L/14 direct JEPA (dim=768, hard curriculum, seed=13)...")
torch.manual_seed(BEST_SEED); np.random.seed(BEST_SEED); random.seed(BEST_SEED)
h_l14 = random.sample(hard_pool, N_TRAIN)
pred_l14_hard = train_jepa768(Zi_l[h_l14], Zt_l[h_l14], epochs=200, seed=BEST_SEED)
torch.save(pred_l14_hard.state_dict(), "/kaggle/working/pred_l14_hard.pt")
print("Saved pred_l14_hard.pt")

Training canonical projection head (L/14 768d → B/32 512d, 300 epochs)...
  ep 100  loss=0.1000
  ep 200  loss=0.0885
  ep 300  loss=0.0873
Saved proj_head.pt

Training L/14 direct JEPA (dim=768, hard curriculum, seed=13)...
Saved pred_l14_hard.pt


In [8]:
def make_foil(caption):
    words = caption.split()
    if len(words) < 5: return None
    sh = words.copy(); att = 0
    while sh == words and att < 20: random.shuffle(sh); att += 1
    return " ".join(sh) if sh != words else None

@torch.no_grad()
def eval_exp2(pred, model, proc, transform_fn=None, n_eval=1000):
    pred.eval().to(DEVICE); errs, labs = [], []
    cands = [i for i in range(len(images)) if i not in train_used]
    random.seed(42); random.shuffle(cands)
    for idx in cands:
        if len(labs) >= n_eval * 2: break
        foil = make_foil(texts[idx])
        if foil is None: continue
        ic  = proc(text=[texts[idx]], images=[images[idx]], return_tensors="pt",
                   padding=True, truncation=True, max_length=77).to(DEVICE)
        if_ = proc(text=[foil],       images=[images[idx]], return_tensors="pt",
                   padding=True, truncation=True, max_length=77).to(DEVICE)
        oc = model(**ic); of = model(**if_)
        zi_c = oc.image_embeds; zt_c = oc.text_embeds
        zi_f = of.image_embeds; zt_f = of.text_embeds
        if transform_fn is not None:
            zi_c = transform_fn(zi_c); zt_c = transform_fn(zt_c)
            zi_f = transform_fn(zi_f); zt_f = transform_fn(zt_f)
        errs += [pred.error(zi_c, zt_c).item(), pred.error(zi_f, zt_f).item()]
        labs += [0, 1]
    pred.cpu()
    return roc_auc_score(labs, errs)

# Random orthogonal projection 768→512
torch.manual_seed(99)
R = torch.linalg.qr(torch.randn(768, 512))[0]
rand_proj_fn    = lambda z: (z @ R.to(z.device))
learned_proj_fn = lambda z: proj_head.to(z.device)(z)

print("EIM Experiment 2 — Encoder Swap via Canonical Projection Head\n")
print("Evaluating 4 conditions (N_eval=1000)...")

auroc_native = eval_exp2(pred_hard, model_b32, proc_b32)
print(f"  1. Native B/32 (no swap)         : {auroc_native:.4f}  (100%)")

auroc_proj = eval_exp2(pred_hard, model_l14, proc_l14, transform_fn=learned_proj_fn)
pct_proj   = 100 * auroc_proj / auroc_native
print(f"  2. Proj (L/14→proj_head→pred)    : {auroc_proj:.4f}  ({pct_proj:.1f}%)")

auroc_rand = eval_exp2(pred_hard, model_l14, proc_l14, transform_fn=rand_proj_fn)
pct_rand   = 100 * auroc_rand / auroc_native
print(f"  3. RandProj (L/14→random→pred)   : {auroc_rand:.4f}  ({pct_rand:.1f}%)")

auroc_l14  = eval_exp2(pred_l14_hard, model_l14, proc_l14)
pct_l14    = 100 * auroc_l14 / auroc_native
print(f"  4. L/14 Direct (JEPA768, no proj): {auroc_l14:.4f}  ({pct_l14:.1f}%)")

print(f"\n{'='*55}")
print(f"  EIM Experiment 2 — Summary")
print(f"{'='*55}")
print(f"  Native B/32       : {auroc_native:.4f}  100.0%")
print(f"  Proj              : {auroc_proj:.4f}  {pct_proj:.1f}%")
print(f"  RandProj          : {auroc_rand:.4f}  {pct_rand:.1f}%")
print(f"  L/14 Direct       : {auroc_l14:.4f}  {pct_l14:.1f}%")

exp2 = dict(native_b32=auroc_native, proj=auroc_proj, rand_proj=auroc_rand,
            l14_direct=auroc_l14, proj_pct=round(pct_proj,1),
            rand_pct=round(pct_rand,1), l14_pct=round(pct_l14,1))
with open("/kaggle/working/exp2_results.json", "w") as fh:
    json.dump(exp2, fh, indent=2)
print("Saved exp2_results.json")

EIM Experiment 2 — Encoder Swap via Canonical Projection Head

Evaluating 4 conditions (N_eval=1000)...
  1. Native B/32 (no swap)         : 0.7272  (100%)
  2. Proj (L/14→proj_head→pred)    : 0.6547  (90.0%)
  3. RandProj (L/14→random→pred)   : 0.4211  (57.9%)
  4. L/14 Direct (JEPA768, no proj): 0.7555  (103.9%)

  EIM Experiment 2 — Summary
  Native B/32       : 0.7272  100.0%
  Proj              : 0.6547  90.0%
  RandProj          : 0.4211  57.9%
  L/14 Direct       : 0.7555  103.9%
Saved exp2_results.json


In [9]:
# EIM Experiment 3: Reactive vs Predictive Routing
# Train 3-layer MLP: image_embed → predicted D-score

class DiffPredictor(nn.Module):
    def __init__(self, in_dim=512):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 256), nn.GELU(),
            nn.Linear(256, 64),    nn.GELU(),
            nn.Linear(64, 1)
        )
    def forward(self, z): return self.net(z).squeeze(-1)

def train_predictor(Zi, D_scores, epochs=200, lr=1e-3, seed=42):
    torch.manual_seed(seed)
    phi = DiffPredictor(Zi.shape[1]).to(DEVICE)
    opt = torch.optim.Adam(phi.parameters(), lr=lr)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    zi  = Zi.to(DEVICE)
    d   = torch.tensor(D_scores, dtype=torch.float32).to(DEVICE)
    for ep in range(1, epochs + 1):
        p    = torch.randperm(len(zi))
        loss = F.mse_loss(phi(zi[p]), d[p])
        opt.zero_grad(); loss.backward(); opt.step(); sch.step()
        if ep % 50 == 0: print(f"  ep {ep}  MSE={loss.item():.5f}")
    return phi.eval().cpu()

print("Training difficulty predictor (image_embed → D-score, 200 epochs)...")
diff_pred = train_predictor(Zi_b, D_b, epochs=200)
torch.save(diff_pred.state_dict(), "/kaggle/working/diff_pred.pt")
print("Saved diff_pred.pt")

Training difficulty predictor (image_embed → D-score, 200 epochs)...
  ep 50  MSE=0.00455
  ep 100  MSE=0.00348
  ep 150  MSE=0.00315
  ep 200  MSE=0.00310
Saved diff_pred.pt


In [10]:
tau_D = np.quantile(D_b, 1 - HARD_FRAC)  # threshold for top 30% = hard

eval_idx = np.array(eval_pool[:1000])

@torch.no_grad()
def eval_predictor(phi, eval_idx):
    phi.eval().to(DEVICE)
    pred_d = phi(Zi_b[eval_idx].to(DEVICE)).cpu().numpy()
    true_d = D_b[eval_idx]
    phi.cpu()

    mae          = float(np.abs(pred_d - true_d).mean())
    rho, _       = spearmanr(pred_d, true_d)
    pred_hard_m  = pred_d >= tau_D
    true_hard_m  = true_d >= tau_D
    agreement    = float((pred_hard_m == true_hard_m).mean())
    auroc        = roc_auc_score(true_hard_m.astype(int), pred_d)

    # per-class breakdown
    th_idx = np.where(true_hard_m)[0]
    te_idx = np.where(~true_hard_m)[0]
    recall      = float((pred_hard_m[th_idx] == True).mean())   # hit rate on hard
    specificity = float((pred_hard_m[te_idx] == False).mean())  # hit rate on easy

    return dict(mae=mae, spearman_rho=float(rho), routing_agreement=agreement,
                auroc=float(auroc), recall_hard=recall, specificity_easy=specificity,
                n_true_hard=int(true_hard_m.sum()), n_true_easy=int((~true_hard_m).sum()))

r = eval_predictor(diff_pred, eval_idx)

print(f"{'='*58}")
print(f"  EIM Experiment 3 — Difficulty Predictor")
print(f"  N_eval={len(eval_idx)}  tau_D={tau_D:.4f} (top {HARD_FRAC*100:.0f}%)")
print(f"{'='*58}")
print(f"  MAE (pred D vs actual D)        : {r['mae']:.4f}")
print(f"  Spearman rho                    : {r['spearman_rho']:.4f}")
print(f"  Routing agreement (overall)     : {r['routing_agreement']*100:.1f}%")
print(f"  AUROC (hard/easy detection)     : {r['auroc']:.4f}")
print(f"{'─'*58}")
print(f"  Per-class breakdown:")
print(f"    True HARD (n={r['n_true_hard']:4d}): recall      = {r['recall_hard']*100:.1f}%")
print(f"    True EASY (n={r['n_true_easy']:4d}): specificity = {r['specificity_easy']*100:.1f}%")
print(f"{'─'*58}")
fn_rate = 1 - r['recall_hard']
print(f"  False negative rate (miss hard) : {fn_rate*100:.1f}%")
print(f"  Savings upper bound             : {r['specificity_easy']*100:.1f}% of easy pairs")

with open("/kaggle/working/exp3_results.json", "w") as fh:
    json.dump(r, fh, indent=2)
print("\nSaved exp3_results.json")

  EIM Experiment 3 — Difficulty Predictor
  N_eval=1000  tau_D=0.7893 (top 30%)
  MAE (pred D vs actual D)        : 0.0405
  Spearman rho                    : 0.1745
  Routing agreement (overall)     : 71.6%
  AUROC (hard/easy detection)     : 0.5965
──────────────────────────────────────────────────────────
  Per-class breakdown:
    True HARD (n= 134): recall      = 32.8%
    True EASY (n= 866): specificity = 77.6%
──────────────────────────────────────────────────────────
  False negative rate (miss hard) : 67.2%
  Savings upper bound             : 77.6% of easy pairs

Saved exp3_results.json


In [11]:
# Re-evaluate on unbiased random sample from full 5000 pairs
rng_idx = np.random.default_rng(42).choice(N_PAIRS, 1000, replace=False)
r_full  = eval_predictor(diff_pred, rng_idx)

print("Re-eval on random 1000 from all 5000 (unbiased distribution):")
print(f"  MAE                : {r_full['mae']:.4f}")
print(f"  Spearman rho       : {r_full['spearman_rho']:.4f}")
print(f"  Routing agreement  : {r_full['routing_agreement']*100:.1f}%")
print(f"  AUROC              : {r_full['auroc']:.4f}")
print(f"  Recall on hard     : {r_full['recall_hard']*100:.1f}%  (n={r_full['n_true_hard']})")
print(f"  Specificity easy   : {r_full['specificity_easy']*100:.1f}%  (n={r_full['n_true_easy']})")

with open("/kaggle/working/exp3_results.json", "w") as fh:
    json.dump({"biased_eval": r, "full_eval": r_full}, fh, indent=2)
print("Updated exp3_results.json")

Re-eval on random 1000 from all 5000 (unbiased distribution):
  MAE                : 0.0442
  Spearman rho       : 0.1285
  Routing agreement  : 63.9%
  AUROC              : 0.5600
  Recall on hard     : 30.1%  (n=306)
  Specificity easy   : 78.8%  (n=694)
Updated exp3_results.json


In [12]:
exp2 = json.load(open("/kaggle/working/exp2_results.json"))
exp3 = json.load(open("/kaggle/working/exp3_results.json"))
r3   = exp3["full_eval"]

print("=" * 60)
print("  NOTEBOOK 2 — EIM EXPERIMENTS 2 & 3 FINAL RESULTS")
print("=" * 60)

print("\n── EIM Exp 2: Encoder Swap ───────────────────────────────")
print(f"  {'Condition':<35} {'AUROC':>7}  {'% native':>8}")
print(f"  {'─'*52}")
print(f"  {'Native B/32 (no swap, baseline)':<35} {exp2['native_b32']:>7.4f}   100.0%")
print(f"  {'Proj (L/14→proj_head→pred_hard)':<35} {exp2['proj']:>7.4f}   {exp2['proj_pct']:>5.1f}%")
print(f"  {'RandProj (L/14→random orth)':<35} {exp2['rand_proj']:>7.4f}   {exp2['rand_pct']:>5.1f}%")
print(f"  {'L/14 Direct (JEPA768, no proj)':<35} {exp2['l14_direct']:>7.4f}   {exp2['l14_pct']:>5.1f}%")
print(f"\n  Key finding: L/14 direct retrain > canonical projection")
print(f"  Proj preserves alignment (90% vs 58% rand) but not best strategy")

print("\n── EIM Exp 3: Difficulty Predictor (unbiased eval) ──────")
print(f"  MAE (pred D vs actual D)        : {r3['mae']:.4f}")
print(f"  Spearman rho                    : {r3['spearman_rho']:.4f}")
print(f"  Routing agreement (overall)     : {r3['routing_agreement']*100:.1f}%")
print(f"  AUROC (hard detection)          : {r3['auroc']:.4f}")
print(f"  Recall on HARD  (n={r3['n_true_hard']:3d})        : {r3['recall_hard']*100:.1f}%")
print(f"  Specificity EASY (n={r3['n_true_easy']:3d})       : {r3['specificity_easy']*100:.1f}%")
print(f"\n  Key finding: image alone is near-chance at predicting")
print(f"  cross-modal difficulty. Reactive routing is necessary.")

final = {"eim_exp2": exp2, "eim_exp3": exp3}
with open("/kaggle/working/nb2_results.json", "w") as fh:
    json.dump(final, fh, indent=2)
print("\nSaved nb2_results.json")

  NOTEBOOK 2 — EIM EXPERIMENTS 2 & 3 FINAL RESULTS

── EIM Exp 2: Encoder Swap ───────────────────────────────
  Condition                             AUROC  % native
  ────────────────────────────────────────────────────
  Native B/32 (no swap, baseline)      0.7272   100.0%
  Proj (L/14→proj_head→pred_hard)      0.6547    90.0%
  RandProj (L/14→random orth)          0.4211    57.9%
  L/14 Direct (JEPA768, no proj)       0.7555   103.9%

  Key finding: L/14 direct retrain > canonical projection
  Proj preserves alignment (90% vs 58% rand) but not best strategy

── EIM Exp 3: Difficulty Predictor (unbiased eval) ──────
  MAE (pred D vs actual D)        : 0.0442
  Spearman rho                    : 0.1285
  Routing agreement (overall)     : 63.9%
  AUROC (hard detection)          : 0.5600
  Recall on HARD  (n=306)        : 30.1%
  Specificity EASY (n=694)       : 78.8%

  Key finding: image alone is near-chance at predicting
  cross-modal difficulty. Reactive routing is necessary.

Saved